# 06 — SBI, IPI, Figure 1 (TP–IP Inversion) & Table 6 Indic Rows

Computes the three novel inference-time diagnostics introduced in **§5 of the paper**:

| Diagnostic | Formula | Ideal | Role |
|---|---|---|---|
| **SBI** | E_i[TP_i / IP_i] | 1.0 | Burden per unit information. Flag ≥ 3.0 |
| **IPI** | \|IP − 1.0\| | 0.0 | Distance from parity; drives zone classification |
| **Computational Tax** | LP × EP | 1.0 | Romanisation overhead (Length × Entropy penalty) |

**Input:** `../../data/processed/Information_parity_outputs_all.xlsx`  
**Outputs:**
- `../../results/fig1_tp_ip_inversion.pdf` — Figure 1 (TP–IP Inversion, two-panel bar chart)
- `../../results/table6_indic_diagnostics.csv` — Table 6 Indic rows (SBI, IPI, Tax, zone labels)

**All numerical values are cross-verified against the paper tables (Table 2, Table 3, Table 6 and Figure 1 annotations).**

## Step 0 — Configuration

In [ ]:
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_FILE   = Path("../../data/processed/Information_parity_outputs_all.xlsx")
RESULTS_DIR = Path("../../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Column names (as in the xlsx) ─────────────────────────────────────────
COL_LANG   = "language"                          # language identifier column

# Native-script TP and IP columns (per sentence)
COL_TP_NAT = "Translation_xlmr_TP"              # TP: tokens-per-word ratio vs English
COL_IP_NAT = "Translation_xlmr_IP"              # IP: NLL compression ratio

# Romanised TP and IP columns
COL_TP_ROM = "Translation_Transliteration_romanized_xlmr_TP"
COL_IP_ROM = "Translation_Transliteration_romanized_xlmr_IP"

# ── Language order (paper convention, sorted by COMET_nat descending) ─────
LANG_ORDER = ["gujarati", "tamil", "malayalam", "marathi", "hindi"]
ISO_MAP    = {"gujarati": "GUJ", "tamil": "TAM",
              "malayalam": "MAL", "marathi": "MAR", "hindi": "HIN"}

# ── Diagnostic thresholds (paper §5, empirically calibrated) ──────────────
SBI_FLAG   = 3.0   # SBI ≥ 3.0 → unreliability flag
IPI_PARITY = 0.05  # IPI < 0.05  → Parity zone
IPI_PARADOX = 0.70 # IPI > 0.70  → Paradox zone; 0.05–0.70 → Burden zone

print("Config loaded. DATA_FILE:", DATA_FILE)

## Step 1 — Load Data

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_excel(DATA_FILE)
print(f"Loaded {len(df):,} rows, {df.shape[1]} columns")

# Validate expected columns exist
for col in [COL_LANG, COL_TP_NAT, COL_IP_NAT, COL_TP_ROM, COL_IP_ROM]:
    assert col in df.columns, f"Missing column: {col}"

# Normalise language names to lowercase
df[COL_LANG] = df[COL_LANG].str.lower().str.strip()
print("Languages in data:", sorted(df[COL_LANG].unique()))
print("Expected:", LANG_ORDER)

## Step 2 — Per-Language Mean TP and IP (native + romanised)

These four values per language feed directly into Figure 1 and the Tax calculation.  
**Paper cross-check** (Figure 1 annotations, §4 Observation 2):
- TAM: TP_nat ≈ 1.32 → TP_rom ≈ 2.35 (+77.9%); IP_nat ≈ 0.545 → IP_rom ≈ 0.174 (−68.0%)

In [ ]:
agg = (
    df.groupby(COL_LANG)[[COL_TP_NAT, COL_IP_NAT, COL_TP_ROM, COL_IP_ROM]]
    .mean()
    .rename(columns={
        COL_TP_NAT: "TP_nat",
        COL_IP_NAT: "IP_nat",
        COL_TP_ROM: "TP_rom",
        COL_IP_ROM: "IP_rom",
    })
    .loc[LANG_ORDER]
    .copy()
)

# Percentage changes (for Figure 1 annotations)
agg["delta_TP_pct"] = (agg["TP_rom"] - agg["TP_nat"]) / agg["TP_nat"] * 100
agg["delta_IP_pct"] = (agg["IP_rom"] - agg["IP_nat"]) / agg["IP_nat"] * 100

print(agg[["TP_nat", "TP_rom", "delta_TP_pct", "IP_nat", "IP_rom", "delta_IP_pct"]].round(3))

# ── Cross-verification against paper (§4, Observation 2) ─────────────────
tam = agg.loc["tamil"]
assert abs(tam["delta_TP_pct"] - 77.9) < 2.0, f"TAM TP% mismatch: {tam['delta_TP_pct']:.1f}"
assert abs(tam["delta_IP_pct"] - (-68.0)) < 2.0, f"TAM IP% mismatch: {tam['delta_IP_pct']:.1f}"
print("\n✓ TAM TP/IP % changes match paper (Figure 1 annotations within 2% tolerance)")

## Step 3 — Compute SBI, IPI, Computational Tax (per sentence → language mean)

Formulas from Table 3 of the paper:
- **SBI** = E_i[TP_i / IP_i] — expectation computed *per sentence* then averaged per language
- **IPI_nat** = |IP_nat − 1.0|; **IPI_rom** = |IP_rom − 1.0|
- **LP** = TP_rom / TP_nat (Length Penalty)  
- **EP** = IP_nat / IP_rom (Entropy Penalty)  
- **Tax** = LP × EP

In [ ]:
# Per-sentence SBI (native and romanised)
df["SBI_nat_sent"] = df[COL_TP_NAT] / df[COL_IP_NAT].replace(0, np.nan)
df["SBI_rom_sent"] = df[COL_TP_ROM] / df[COL_IP_ROM].replace(0, np.nan)

# Language-mean SBI
sbi_nat = df.groupby(COL_LANG)["SBI_nat_sent"].mean().loc[LANG_ORDER]
sbi_rom = df.groupby(COL_LANG)["SBI_rom_sent"].mean().loc[LANG_ORDER]

# IPI from mean IP
ipi_nat = (agg["IP_nat"] - 1.0).abs()
ipi_rom = (agg["IP_rom"] - 1.0).abs()

# Computational Tax components (from language-mean TP/IP)
lp  = agg["TP_rom"] / agg["TP_nat"]   # Length Penalty
ep  = agg["IP_nat"] / agg["IP_rom"]   # Entropy Penalty
tax = lp * ep

# Zone classification (§5)
def zone(ipi):
    if ipi < IPI_PARITY:  return "Parity"
    if ipi > IPI_PARADOX: return "Paradox"
    return "Burden"

zone_nat = ipi_nat.map(zone)
zone_rom = ipi_rom.map(zone)

# Assemble Table 6
table6 = pd.DataFrame({
    "ISO":       [ISO_MAP[l] for l in LANG_ORDER],
    "TP_nat":    agg["TP_nat"].values,
    "IP_nat":    agg["IP_nat"].values,
    "SBI_nat":   sbi_nat.values,
    "IPI_nat":   ipi_nat.values,
    "Zone_nat":  zone_nat.values,
    "TP_rom":    agg["TP_rom"].values,
    "IP_rom":    agg["IP_rom"].values,
    "SBI_rom":   sbi_rom.values,
    "IPI_rom":   ipi_rom.values,
    "Zone_rom":  zone_rom.values,
    "LP":        lp.values,
    "EP":        ep.values,
    "Tax":       tax.values,
}, index=LANG_ORDER)

print(table6.to_string())

## Step 4 — Cross-Verify Against Paper Values

All assertions check computed values against numbers explicitly stated in the paper.

In [ ]:
# ── Paper §5: SBI_nat reproduces COMET ranking exactly (ρ=1.000) ──────────
# Expected COMET_nat ranking (Table 2): GUJ > TAM > MAL > MAR ≈ HIN
# SBI_nat order should match → GUJ highest SBI_nat
assert table6.loc["gujarati", "SBI_nat"] > table6.loc["hindi", "SBI_nat"], \
    "SBI_nat: GUJ should be higher burden than HIN"

# ── Paper §5: SBI_nat = 3.45 for GUJ ─────────────────────────────────────
guj_sbi = table6.loc["gujarati", "SBI_nat"]
assert abs(guj_sbi - 3.45) < 0.15, f"GUJ SBI_nat = {guj_sbi:.3f}, expected ≈3.45"
print(f"✓ GUJ SBI_nat = {guj_sbi:.3f} (paper: 3.45)")

# ── Paper §5: SBI flag ≥ 3.0 → GUJ must be flagged ──────────────────────
assert table6.loc["gujarati", "SBI_nat"] >= SBI_FLAG, "GUJ should trigger SBI flag"
print(f"✓ GUJ SBI_nat ≥ {SBI_FLAG} flag triggered")

# ── Paper §5: Tax range 1.75x (GUJ) to 5.57x (TAM) ──────────────────────
guj_tax = table6.loc["gujarati", "Tax"]
tam_tax = table6.loc["tamil",    "Tax"]
assert abs(guj_tax - 1.75) < 0.15, f"GUJ Tax = {guj_tax:.3f}, expected ≈1.75"
assert abs(tam_tax - 5.57) < 0.20, f"TAM Tax = {tam_tax:.3f}, expected ≈5.57"
print(f"✓ GUJ Tax = {guj_tax:.2f}x (paper: 1.75x)")
print(f"✓ TAM Tax = {tam_tax:.2f}x (paper: 5.57x)")

# ── Paper §5: Every Indic language crosses into Paradox under romanisation
for lang in LANG_ORDER:
    z = table6.loc[lang, "Zone_rom"]
    assert z == "Paradox", f"{lang} Zone_rom = {z}, expected Paradox"
print("✓ All five Indic languages in Paradox zone under romanisation")

# ── Paper §5: Every Indic language in Burden zone under native script
for lang in LANG_ORDER:
    z = table6.loc[lang, "Zone_nat"]
    assert z == "Burden", f"{lang} Zone_nat = {z}, expected Burden"
print("✓ All five Indic languages in Burden zone under native script")

# ── IP_nat for GUJ (paper §5: IP_nat = 0.438) ─────────────────────────────
guj_ip = table6.loc["gujarati", "IP_nat"]
assert abs(guj_ip - 0.438) < 0.02, f"GUJ IP_nat = {guj_ip:.3f}, expected ≈0.438"
print(f"✓ GUJ IP_nat = {guj_ip:.3f} (paper: 0.438)")

print("\n── All cross-verification assertions passed ──")

## Step 5 — Figure 1: The TP–IP Inversion (two-panel bar chart)

Left panel: IP native vs romanised per language (IP collapses under romanisation).  
Right panel: TP native vs romanised per language (TP inflates under romanisation).  
TAM annotated with −68.0% and +77.9% per paper.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

iso_labels = [ISO_MAP[l] for l in LANG_ORDER]
x = np.arange(len(LANG_ORDER))
w = 0.35

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
fig.suptitle("Figure 1: The TP–IP Inversion", fontsize=13, fontweight="bold", y=1.01)

# ── LEFT: IP panel ────────────────────────────────────────────────────────
ax = axes[0]
bars_nat = ax.bar(x - w/2, agg["IP_nat"].values, w, label="Native script",
                  color="#2c7bb6", zorder=3)
bars_rom = ax.bar(x + w/2, agg["IP_rom"].values, w, label="Romanised",
                  color="#d7191c", zorder=3)
ax.set_title("Information Parity (IP)\nIP collapses under romanisation",
             fontsize=10)
ax.set_ylabel("Mean IP (higher = more semantic density per token)", fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(iso_labels, fontsize=10)
ax.set_ylim(0, 1.0)
ax.axhline(1.0, color="grey", linewidth=0.8, linestyle="--", label="English parity")
ax.yaxis.set_minor_locator(mticker.MultipleLocator(0.05))
ax.grid(axis="y", linestyle=":", linewidth=0.5, zorder=0)
ax.legend(fontsize=8)

# Annotate TAM with paper value
tam_idx = LANG_ORDER.index("tamil")
ax.annotate(
    f"{agg.loc['tamil', 'delta_IP_pct']:.1f}%",
    xy=(tam_idx + w/2, agg.loc["tamil", "IP_rom"]),
    xytext=(tam_idx + w/2 + 0.05, agg.loc["tamil", "IP_rom"] + 0.06),
    fontsize=8, color="#d7191c",
    arrowprops=dict(arrowstyle="->", color="#d7191c", lw=0.8),
)

# ── RIGHT: TP panel ───────────────────────────────────────────────────────
ax = axes[1]
ax.bar(x - w/2, agg["TP_nat"].values, w, label="Native script",
       color="#2c7bb6", zorder=3)
ax.bar(x + w/2, agg["TP_rom"].values, w, label="Romanised",
       color="#d7191c", zorder=3)
ax.set_title("Tokenization Parity (TP)\nTP inflates under romanisation",
             fontsize=10)
ax.set_ylabel("Mean TP (lower = less fragmentation)", fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(iso_labels, fontsize=10)
ax.axhline(1.0, color="grey", linewidth=0.8, linestyle="--", label="English parity")
ax.yaxis.set_minor_locator(mticker.MultipleLocator(0.1))
ax.grid(axis="y", linestyle=":", linewidth=0.5, zorder=0)
ax.legend(fontsize=8)

# Annotate TAM with paper value
ax.annotate(
    f"+{agg.loc['tamil', 'delta_TP_pct']:.1f}%",
    xy=(tam_idx + w/2, agg.loc["tamil", "TP_rom"]),
    xytext=(tam_idx + w/2 + 0.05, agg.loc["tamil", "TP_rom"] + 0.1),
    fontsize=8, color="#d7191c",
    arrowprops=dict(arrowstyle="->", color="#d7191c", lw=0.8),
)

fig.tight_layout()
out_fig = RESULTS_DIR / "fig1_tp_ip_inversion.pdf"
fig.savefig(out_fig, bbox_inches="tight", dpi=300)
print(f"Figure saved → {out_fig}")
plt.show()

## Step 6 — Save Table 6 Indic Rows

In [ ]:
out_csv = RESULTS_DIR / "table6_indic_diagnostics.csv"
table6.to_csv(out_csv)

# Pretty-print for notebook display
display_cols = [
    "ISO",
    "TP_nat", "IP_nat", "SBI_nat", "IPI_nat", "Zone_nat",
    "TP_rom", "IP_rom", "SBI_rom", "IPI_rom", "Zone_rom",
    "LP", "EP", "Tax",
]
print("Table 6 — Indic rows (SBI, IPI, Computational Tax):")
print(table6[display_cols].round(3).to_string())
print(f"\nCSV saved → {out_csv}")

## Step 7 — Summary Printout (Spot-Check)

Final sanity check: print the key numbers the paper cites to confirm they are reproduced.

In [ ]:
print("=" * 60)
print("Key numbers vs paper claims")
print("=" * 60)

rows = [
    ("GUJ SBI_nat",   table6.loc["gujarati", "SBI_nat"],  "3.45",  "§5"),
    ("GUJ IP_nat",    table6.loc["gujarati", "IP_nat"],   "0.438", "§5"),
    ("TAM delta_TP%", agg.loc["tamil", "delta_TP_pct"],   "+77.9", "Fig 1"),
    ("TAM delta_IP%", agg.loc["tamil", "delta_IP_pct"],   "-68.0", "Fig 1"),
    ("GUJ Tax",       table6.loc["gujarati", "Tax"],       "1.75×", "§5"),
    ("TAM Tax",       table6.loc["tamil",    "Tax"],       "5.57×", "§5"),
]
for name, computed, paper_val, ref in rows:
    print(f"{name:<20} computed={computed:>8.3f}   paper={paper_val:>6}   ({ref})")

print()
print("Zone assignments (native script — all should be Burden):")
for lang in LANG_ORDER:
    iso = ISO_MAP[lang]
    z_nat = table6.loc[lang, "Zone_nat"]
    z_rom = table6.loc[lang, "Zone_rom"]
    print(f"  {iso}: native={z_nat:<8} romanised={z_rom}")